In [5]:
import pandas as pd
import glob

In [6]:
print('Hello')

Hello


In [12]:
csv_files = glob.glob("*.csv")

def analyze_dataset(data):
    # Defining the groups based on row ranges
    # Note: data.iloc[start:stop] handles the slicing
    groups = {
        "combinational_basic": data.iloc[0:80],
        "sequential_basic":    data.iloc[80:160],
        "fsm":                 data.iloc[160:190],
        "industry":            data.iloc[-10:] # Gets the last 10 rows
    }

    results = {}

    for name, df in groups.items():
        if not df.empty:
            # Calculate % where iverilog_output is "OK"
            success_pct = (df['iverilog_output'] == 'OK').mean() * 100
            
            # Calculate average of time(s)
            avg_time = df['time(s)'].mean()
            
            results[name] = {
                "Success Rate (%)": round(success_pct, 2),
                "Avg Time (s)": round(avg_time, 4)
            }
    
    # Convert results to a readable DataFrame
    summary_df = pd.DataFrame(results).T
    return summary_df

In [13]:
# Usage:
for csv_file_datapath in csv_files:
    data = pd.read_csv(csv_file_datapath)
    print(csv_file_datapath, f'total examples: {data.shape[0]}', f"Average success rate: {round((data['iverilog_output'] == 'OK').mean() * 100 , 2)}%" )
    print(analyze_dataset(data))
    print('---------------------------------------------------')

sv_results_yi-coder_9b_3.csv total examples: 194 Average success rate: 79.38%
                     Success Rate (%)  Avg Time (s)
combinational_basic             82.50        4.4468
sequential_basic                92.50       11.1624
fsm                             43.33       37.1658
industry                        50.00       58.8430
---------------------------------------------------
sv_results_deepseek-coder_33b_3_2.csv total examples: 5 Average success rate: 0.0%
                     Success Rate (%)  Avg Time (s)
combinational_basic               0.0      219.4225
industry                          0.0      219.4225
---------------------------------------------------
sv_results_qwen2.5-coder_7b_3.csv total examples: 187 Average success rate: 95.19%
                     Success Rate (%)  Avg Time (s)
combinational_basic            100.00        3.1339
sequential_basic                97.50       10.5731
fsm                             74.07       24.7953
industry                    

In [ ]:
import pandas as pd
import os

# 1. Load the master dataset and get the unique modules as a set for speed
master_df = pd.read_csv('dataset.csv')
master_modules = set(master_df['modules'].dropna().unique())

# 2. Identify all other CSV files in the directory
all_files = [f for f in os.listdir('.') if f.endswith('.csv') and f != 'dataset.csv']

results = []

for file in all_files:
    try:
        # Load the current comparison file
        current_df = pd.read_csv(file)
        
        if 'original_code' in current_df.columns:
            # Get unique codes from the current file
            current_codes = set(current_df['original_code'].dropna().unique())
            
            # 3. Find elements in current_codes that are NOT in master_modules
            different_elements = current_codes - master_modules
            diff_count = len(different_elements)
            
            results.append({'filename': file, 'different_rows_count': diff_count})
        else:
            print(f"Skipping {file}: Column 'original_code' not found.")
            
    except Exception as e:
        print(f"Error processing {file}: {e}")

# 4. Create a summary DataFrame and display/save
summary_df = pd.DataFrame(results)
print("\nComparison Summary:")
print(summary_df)

# Optional: Save the summary to a file
# summary_df.to_csv('comparison_results.csv', index=False)